# Task 1: Stochastic SIR Simulation

Generate stochastic SIR epidemics using Gillespie's SSA.
- Validate against deterministic ODE for large N
- Generate full dataset across parameter grid
- Compute ensemble means and visualize

In [ ]:
!git clone https://github.com/invi-bhagyesh/sira.git 2>/dev/null || echo "already cloned"
%cd sira
!pip install -q -r requirements.txt

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.simulation.gillespie import SIRParams, gillespie_sir, simulate_ensemble, deterministic_sir
from src.simulation.dataset import generate_dataset, load_dataset, train_val_test_split

## 1. Validate Gillespie SSA Against Deterministic ODE

For large N, the ensemble mean of stochastic trajectories should converge to the deterministic solution.

In [ ]:
params = SIRParams(beta=0.5, gamma=0.2, N=10000, I0=10)
print(f"R0 = {params.R0:.2f}")

# deterministic solution
t_det, s_det, i_det, r_det = deterministic_sir(params, t_max=60)

# stochastic ensemble
result = simulate_ensemble(params, M=100, t_max=60)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
labels = ['S (Susceptible)', 'I (Infected)', 'R (Removed)']
det_curves = [s_det, i_det, r_det]
mean_curves = [result['mean_s'], result['mean_i'], result['mean_r']]
std_curves = [result['std_s'], result['std_i'], result['std_r']]

for ax, label, det, mean, std in zip(axes, labels, det_curves, mean_curves, std_curves):
    ax.plot(t_det, det, 'k--', lw=2, label='Deterministic ODE')
    ax.plot(result['t_grid'], mean, 'b-', lw=1.5, label='Stochastic mean')
    ax.fill_between(result['t_grid'], mean - std, mean + std, alpha=0.2)
    ax.set_xlabel('Time')
    ax.set_ylabel('Fraction of population')
    ax.set_title(label)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle(f'Validation: beta={params.beta}, gamma={params.gamma}, N={params.N}', fontsize=12)
plt.tight_layout()
plt.show()

# quantify convergence
mae_s = np.mean(np.abs(result['mean_s'] - s_det[:len(result['mean_s'])]))
mae_i = np.mean(np.abs(result['mean_i'] - i_det[:len(result['mean_i'])]))
print(f'MAE (stochastic mean vs deterministic): S={mae_s:.4f}, I={mae_i:.4f}')

## 2. Visualize Stochastic Variability

Show individual trajectories alongside the ensemble mean for different population sizes.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, N in zip(axes, [500, 1000, 5000, 10000]):
    params = SIRParams(beta=0.5, gamma=0.2, N=N, I0=5)
    result = simulate_ensemble(params, M=50, t_max=60)

    for m in range(min(20, 50)):
        ax.plot(result['t_grid'], result['all_i'][m], alpha=0.15, color='blue', lw=0.5)
    ax.plot(result['t_grid'], result['mean_i'], 'r-', lw=2, label='Mean')
    ax.set_title(f'N = {N}')
    ax.set_xlabel('Time')
    ax.set_ylabel('I/N')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Stochastic variability decreases with population size', fontsize=12)
plt.tight_layout()
plt.show()

## 3. Effect of R0

Show how epidemic dynamics change with R0 = beta/gamma.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
gamma = 0.2

for ax, beta in zip(axes, [0.15, 0.4, 0.8]):
    params = SIRParams(beta=beta, gamma=gamma, N=5000, I0=5)
    result = simulate_ensemble(params, M=100, t_max=80)
    t_det, s_det, i_det, r_det = deterministic_sir(params, t_max=80)

    ax.plot(result['t_grid'], result['mean_s'], label='S')
    ax.plot(result['t_grid'], result['mean_i'], label='I')
    ax.plot(result['t_grid'], result['mean_r'], label='R')
    ax.set_title(f'R0 = {params.R0:.2f} (beta={beta})')
    ax.set_xlabel('Time')
    ax.set_ylabel('Fraction')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Epidemic dynamics across R0 regimes', fontsize=12)
plt.tight_layout()
plt.show()

## 4. Generate Full Dataset

1,200 parameter combinations x 100 realizations = 120,000 trajectories.
Filter out extinction-dominated combinations.

In [ ]:
import os

if os.path.exists('data/sir_dataset.npz'):
    print('Dataset already exists, loading...')
    data = load_dataset('data/sir_dataset.npz')
else:
    data = generate_dataset(save_dir='data')

print(f"Dataset shape: {data['mean_s'].shape}")
print(f"Parameters shape: {data['params'].shape}")
print(f"Time grid: {len(data['t_grid'])} points, [0, {data['t_grid'][-1]}]")
print(f"\nParameter ranges:")
print(f"  beta:  [{data['params'][:, 0].min():.2f}, {data['params'][:, 0].max():.2f}]")
print(f"  gamma: [{data['params'][:, 1].min():.2f}, {data['params'][:, 1].max():.2f}]")
print(f"  N:     {sorted(set(data['params'][:, 2].astype(int)))}")
print(f"  I0:    {sorted(set(data['params'][:, 3].astype(int)))}")

In [ ]:
train, val, test = train_val_test_split(data)
print(f'Train: {len(train["params"])}, Val: {len(val["params"])}, Test: {len(test["params"])}')